# fit / predict / score — единый sklearn-API

**II Саммит талантов · трек ИИ · Д2П2 · Иван Эйдлин**

Цель ноутбука: показать, что у **любой** модели в `sklearn` один и тот же интерфейс:

```
model.fit(X_train, y_train)   # обучить
model.predict(X_test)         # предсказать
model.score(X_test, y_test)   # оценить
```

Меняется только модель. Код — тот же.

## 1. Загружаем данные

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame
df.head()

In [ ]:
df.describe()

In [ ]:
df['target'].value_counts()

## 2. Разделяем на train и test

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('train:', X_train.shape, 'test:', X_test.shape)

## 3. Одна функция — любая модель

Магия `sklearn` в том, что к каждой модели можно применить один и тот же код.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

def run(model, name):
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f'{name:20s}  accuracy = {acc:.3f}')
    return model

run(LogisticRegression(max_iter=1000), 'LogisticRegression')
run(KNeighborsClassifier(n_neighbors=5), 'KNN (k=5)')
run(RandomForestClassifier(random_state=42), 'RandomForest');

> Обратите внимание: мы **поменяли модель**, но **не поменяли код**. Это и есть единый `fit / predict / score`-интерфейс.

## 4. Предсказание на новом объекте

In [ ]:
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)

new_flower = [[5.1, 3.5, 1.4, 0.2]]
pred = model.predict(new_flower)[0]
proba = model.predict_proba(new_flower)[0]

print('класс:', iris.target_names[pred])
for name, p in zip(iris.target_names, proba):
    print(f'  {name:12s}  P = {p:.2f}')

## 5. Что сказать классу

1. `sklearn` — это «гайка одного ключа». Выучил интерфейс — умеешь 100 моделей.
2. Модель — это **чёрный ящик с двумя кнопками**: `fit` (учись) и `predict` (отвечай).
3. Сравнивать модели — это просто сравнивать их `score` на одних и тех же `X_test` и `y_test`.
4. Главный риск — взять ту же выборку для обучения и проверки. Тогда модель получит 100%, но не научится ничему полезному.